# Session 2, Live Demo: Variation, Probability & Displaying Data

**Instructor demo notebook. Drive it conversationally, and make the room *predict before you compute*.**

Three legs today: how spread out the data is (variation and risk), the language of chance (probability), and how to show data (the right chart for the question). The normal curve, z-scores and the Central Limit Theorem come next week in Session 3.

> No file to download. We simulate the business datasets in code so the numbers are stable.

In [ ]:
# --- Generate the data (reproducible, fixed seed) ------------------
# No file to download: we simulate a few business datasets in code.
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

# 1) A fund's ANNUAL returns: historically about 8% average, 12% sd.
#    We simulate 1,000 hypothetical years to see the whole distribution.
returns_pct = rng.normal(8, 12, size=1000)          # already in %

# 2) Two funds with the SAME average return but different risk.
fund_A = rng.normal(8, 6, size=1000)                # steady
fund_B = rng.normal(8, 14, size=1000)               # volatile

# 3) Invoice amounts (euros): right-skewed, like most financial amounts.
#    Most invoices are small; a few are very large.
invoices = rng.lognormal(mean=7.6, sigma=0.8, size=1000)   # median ~ EUR 2,000

print(f"returns:  mean {returns_pct.mean():.1f}%   sd {returns_pct.std():.1f}%")
print(f"fund A:   mean {fund_A.mean():.1f}%   sd {fund_A.std():.1f}%")
print(f"fund B:   mean {fund_B.mean():.1f}%   sd {fund_B.std():.1f}%")
print(f"invoices: mean EUR {invoices.mean():,.0f}   median EUR {np.median(invoices):,.0f}")

## Step 0, the "champion flipper" experiment  (run it in the room first)

We just did this live: everyone flips, wrong calls sit down, repeat until one "champion" survives about 6 rounds. **Predict before you compute:** in a room of 75, how many people will flip the same call 6 times in a row, by pure luck? Commit to a number, then run the cell.

Two rules in play: independent events **multiply** (P = 1/2 each round), and we have *many* people, so rare things happen to *someone*.

In [ ]:
n_people, rounds = 75, 6
p_survive = 0.5 ** rounds                 # (1/2)^6 = 1/64
expected = n_people * p_survive
print(f'P(survive {rounds} rounds) = {p_survive:.4f}  (about 1 in {1/p_survive:.0f})')
print(f'Expected champions in {n_people} people: {expected:.1f}')

# Simulate the whole tournament 10,000 times: how many champions usually emerge?
rng_coin = np.random.default_rng(1)
sims = rng_coin.random((10000, n_people, rounds)) < 0.5   # True = called it right
champions = (sims.all(axis=2)).sum(axis=1)               # survived all rounds
print(f'\nAcross 10,000 rooms: average {champions.mean():.2f} champions, ',
      f'range {champions.min()}-{champions.max()}')

**The lesson:** about 1 champion is *guaranteed by luck* in a room this size, nobody is gifted. That is **noise dressed up as signal.** Two links to carry forward:

- *Today:* rare-by-luck is real. A value 2 standard deviations out is uncommon, not impossible. **Unusual is not proof.**
- *Session 4:* this is exactly what a **p-value** asks, "champion flipper, or a real effect?" An auditor asks the same of every anomaly.

## Step 1, variation: same average, very different risk

Two funds, both averaging about 8% a year. The **centre cannot tell them apart**, so a retiree who only looks at the average learns nothing about the risk. The number that separates them is the **spread**.

- **Standard deviation (SD):** how far a typical year sits from the average, in the data's own units.
- **Coefficient of variation (CV) = SD / mean:** risk per unit of return, so you can compare funds on different scales.

In [ ]:
import matplotlib.pyplot as plt
for name, x in [('Fund A (steady)', fund_A), ('Fund B (volatile)', fund_B)]:
    m, s = x.mean(), x.std(ddof=1)
    print(f'{name:18}  mean {m:5.1f}%   sd {s:5.2f}   CV {s/m*100:5.1f}%')

plt.figure(figsize=(9, 4.2))
bins = np.linspace(-45, 60, 44)
plt.hist(fund_A, bins=bins, color='#3E8E5A', alpha=0.6, label='Fund A: steady')
plt.hist(fund_B, bins=bins, color='#C0392B', alpha=0.45, label='Fund B: volatile')
plt.axvline(8, color='#1F497D', ls='--', lw=2, label='both average ~8%')
plt.yticks([]); plt.xlabel('annual return (%)')
plt.title('Same average, very different spread'); plt.legend(); plt.show()

**Talk track:** both average about 8%, but Fund B swings far wider, so its SD and CV are much larger. Finance calls that spread *volatility*, and volatility is risk. Ask Claude *"which fund suits a risk-averse client, and which number proves it?"* and check that it reaches for the spread, not the average.

## Step 2, a distribution is just *"what values, how often"*

Describe any distribution by three things: **shape** (symmetric or skewed?), **centre** (mean or median), **spread** (the SD from Step 1). Watch what *skew* does to the mean versus the median.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(returns_pct, bins=30, color='#1F497D', alpha=0.85)
ax[0].set_title('Fund annual returns (%): roughly symmetric')
ax[0].set_xlabel('annual return (%)')
ax[1].hist(invoices, bins=40, color='#0D73B8', alpha=0.85)
ax[1].set_title('Invoice amounts (EUR): right-skewed')
ax[1].set_xlabel('invoice amount (EUR)')
plt.tight_layout(); plt.show()

for name, x in [('returns (%)', returns_pct), ('invoices (EUR)', invoices)]:
    print(f'{name:14}  mean {x.mean():10.1f}   median {np.median(x):10.1f}')

**Talk track:** the returns are roughly symmetric, so mean and median agree. The invoices are right-skewed, so a few huge invoices drag the **mean above the median**. An auditor who quotes the *mean* of a skewed population is already half-wrong. Ask Claude *which* average to report, and why.

## Step 3, how to *see* a distribution: histogram and box plot

The **box plot** packs the five-number summary into one shape: the box is the middle 50% (Q1 to Q3), the line inside is the median, the whiskers reach the rest, and stray points are outliers. It is the fastest way to compare spread and spot a suspicious tail.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].boxplot([fund_A, fund_B], orientation='vertical', widths=0.5,
              tick_labels=['Fund A\n(steady)', 'Fund B\n(volatile)'])
ax[0].set_ylabel('annual return (%)'); ax[0].set_title('Same centre, different box width')
ax[1].boxplot(invoices, orientation='horizontal', widths=0.5)
ax[1].set_yticks([]); ax[1].set_xlabel('invoice amount (EUR)')
ax[1].set_title('A long right tail = skew + outliers')
plt.tight_layout(); plt.show()

# The five-number summary behind the box, straight from numpy.
q1, med, q3 = np.percentile(invoices, [25, 50, 75])
print(f'invoices:  Q1 {q1:,.0f}   median {med:,.0f}   Q3 {q3:,.0f}   IQR {q3-q1:,.0f}')

**Claude beat:** describe one of these box plots to Claude and ask *"is this data skewed, and would a box plot or a histogram tell the client more?"* Then check its answer against the picture. Catching a confident-but-wrong reading is the skill.

## Wrap, why today matters

Three tools, one habit. **Variation** puts a number on risk (the SD and the CV). **Probability** separates a lucky streak from a real effect. **Displaying data** picks the chart that answers the question without misleading. All three run under audit work: sizing risk, weighing an anomaly, and reporting a result honestly.

**Next week (Session 3):** the **normal curve**, **z-scores**, and the one idea (the **Central Limit Theorem**) that lets an auditor test a *sample* and speak for the *whole population*.

Habit check: did you **predict before you computed**?